# Tratamento de Dados — Teste Técnico Analista de BI Pleno

Cenário: utilização de planos de saúde. Três arquivos de origem:

- `Beneficiarios.csv` — id_beneficiario, sexo, idade, uf, id_operadora
- `Atendimentos.csv` — id_atendimento, id_beneficiario, data_atendimento, especialidade, valor
- `Operadoras.csv` — id_operadora, operadora

Este notebook faz o diagnóstico de qualidade dos dados, aplica as regras de tratamento definidas
(com justificativa) e exporta as dimensões e a fato do modelo dimensional em estrela para
`../dados/tratados/`.

## 1. Imports e configuração

In [1]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

from pathlib import Path

PASTA_BRUTOS = Path('../dados/brutos')
PASTA_TRATADOS = Path('../dados/tratados')
PASTA_TRATADOS.mkdir(parents=True, exist_ok=True)

HOJE = pd.Timestamp('2026-09-05')  # data de referência da execução do teste
DATA_MIN_VALIDA = pd.Timestamp('2015-01-01')  # limite inferior plausível para atendimentos

## 2. Leitura dos arquivos de origem

Os três CSVs estão em UTF-8 (confirmado via inspeção dos bytes brutos dos campos acentuados —
ex.: `especialidade`). Todas as colunas são lidas como `object`/texto nesta etapa (sem `dtype`
forçado), justamente para não mascarar valores inválidos que um parser numérico/data
descartaria silenciosamente (`NaN`) antes do diagnóstico.

In [2]:
df_benef_raw = pd.read_csv(PASTA_BRUTOS / 'Beneficiarios.csv', dtype=str)
df_atend_raw = pd.read_csv(PASTA_BRUTOS / 'Atendimentos.csv', dtype=str)
df_oper_raw = pd.read_csv(PASTA_BRUTOS / 'Operadoras.csv', dtype=str)

print('Beneficiarios:', df_benef_raw.shape)
print('Atendimentos :', df_atend_raw.shape)
print('Operadoras   :', df_oper_raw.shape)

Beneficiarios: (10000, 5)
Atendimentos : (100000, 5)
Operadoras   : (300, 2)


## 3. Diagnóstico de qualidade dos dados

Análise coluna a coluna, registrando tipo e quantidade de problema encontrado antes de decidir
a regra de tratamento.

### 3.1 Beneficiarios.csv

In [3]:
print('--- Nulos por coluna ---')
print(df_benef_raw.isna().sum())

print('\n--- Duplicidade ---')
print('Linhas 100% duplicadas:', df_benef_raw.duplicated().sum())
print('id_beneficiario duplicado:', df_benef_raw['id_beneficiario'].duplicated().sum())

print('\n--- sexo: valores distintos ---')
print(df_benef_raw['sexo'].value_counts(dropna=False))

idade_num = pd.to_numeric(df_benef_raw['idade'], errors='coerce')
print('\n--- idade ---')
print('Não numéricos:', df_benef_raw.loc[idade_num.isna() & df_benef_raw['idade'].notna(), 'idade'].unique())
print('Fora da faixa plausível (<=0 ou >110):', ((idade_num <= 0) | (idade_num > 110)).sum())

print('\n--- uf: valores distintos ---')
print(sorted(df_benef_raw['uf'].dropna().str.strip().str.upper().unique().tolist()))

id_op_num = pd.to_numeric(df_benef_raw['id_operadora'], errors='coerce')
print('\n--- id_operadora (FK p/ Operadoras) ---')
print('Não numéricos:', df_benef_raw.loc[id_op_num.isna() & df_benef_raw['id_operadora'].notna(), 'id_operadora'].unique())
print('Negativos:', (id_op_num < 0).sum())

--- Nulos por coluna ---
id_beneficiario      0
sexo               237
idade              200
uf                 229
id_operadora       200
dtype: int64

--- Duplicidade ---
Linhas 100% duplicadas: 112
id_beneficiario duplicado: 200

--- sexo: valores distintos ---
sexo
F            4812
M            4788
NaN           237
m              37
Feminino       34
X              31
f              31
Masculino      30
Name: count, dtype: int64

--- idade ---
Não numéricos: ['dez']
Fora da faixa plausível (<=0 ou >110): 290

--- uf: valores distintos ---
['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MINAS GERAIS', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RIO', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO', 'XX']

--- id_operadora (FK p/ Operadoras) ---
Não numéricos: ['ABC']
Negativos: 77


**Achados — Beneficiarios.csv**

| Campo | Problema | Qtd. |
|---|---|---|
| (linha inteira) | duplicidade total | 112 linhas |
| id_beneficiario | id duplicado (registros distintos) | 200 |
| sexo | nulo | 237 |
| sexo | variações de grafia (`m`, `f`, `Masculino`, `Feminino`, `X`) | 132 |
| idade | nulo | 200 |
| idade | não numérico (`"dez"`) | 1 |
| idade | fora da faixa plausível (≤0 ou >110) | ~290 |
| uf | nulo | 229 |
| uf | por extenso (`MINAS GERAIS`) ou inválida (`XX`, `RIO`) | — |
| id_operadora | nulo | 200 |
| id_operadora | não numérico (`ABC`) ou negativo | ~77+ |

**Decisões de tratamento (Beneficiarios):**
1. Remover linhas 100% duplicadas.
2. Para `id_beneficiario` duplicado (após remoção acima), manter a **primeira ocorrência** — não há
   coluna de data/versão que indique qual registro é o mais recente, então a regra é determinística
   e documentada aqui.
3. `sexo`: normalizar para `M`, `F` ou `Não Informado` (nulo, `X`, ou qualquer valor fora do
   domínio esperado). Não há como inferir sexo a partir de outros campos.
4. `idade`: converter para numérico; valores não numéricos, nulos ou fora de `[0, 110]` viram
   `NaN` e o beneficiário é classificado na faixa etária `Não Informada` — optei por **não
   imputar** (média/mediana) para não introduzir viés artificial nos indicadores.
5. `uf`: normalizar case/espaços; mapear grafias por extenso conhecidas (`MINAS GERAIS` → `MG`);
   valores não mapeáveis (`XX`, `RIO`, nulo) → `NI` (Não Informado).
6. `id_operadora`: valores nulos, negativos ou não numéricos não têm informação de apoio (ao
   contrário de `Operadoras.csv`, não há nome de operadora para reconciliar) → aponta para o
   registro sentinela `-1 = Operadora Não Identificada` na dimensão de operadoras.

In [4]:
df_b = df_benef_raw.drop_duplicates().copy()
df_b = df_b.drop_duplicates(subset='id_beneficiario', keep='first')
df_b['id_beneficiario'] = df_b['id_beneficiario'].astype('int64')

# sexo
mapa_sexo = {'F': 'F', 'M': 'M', 'f': 'F', 'm': 'M', 'Feminino': 'F', 'Masculino': 'M'}
df_b['sexo'] = df_b['sexo'].map(mapa_sexo).fillna('Não Informado')

# idade -> numérica válida + faixa etária
df_b['idade'] = pd.to_numeric(df_b['idade'], errors='coerce')
df_b.loc[(df_b['idade'] <= 0) | (df_b['idade'] > 110), 'idade'] = np.nan

def faixa_etaria(idade):
    if pd.isna(idade):
        return 'Não Informada'
    if idade <= 18:
        return '0-18'
    if idade <= 30:
        return '19-30'
    if idade <= 45:
        return '31-45'
    if idade <= 60:
        return '46-60'
    return '60+'

df_b['faixa_etaria'] = df_b['idade'].apply(faixa_etaria)

# uf
mapa_uf_extenso = {'MINAS GERAIS': 'MG'}
ufs_validas = {
    'AC','AL','AM','AP','BA','CE','DF','ES','GO','MA','MG','MS','MT','PA','PB',
    'PE','PI','PR','RJ','RN','RO','RR','RS','SC','SE','SP','TO'
}
df_b['uf'] = df_b['uf'].str.strip().str.upper()
df_b['uf'] = df_b['uf'].replace(mapa_uf_extenso)
df_b.loc[~df_b['uf'].isin(ufs_validas), 'uf'] = 'NI'
df_b['uf'] = df_b['uf'].fillna('NI')

# id_operadora
df_b['id_operadora'] = pd.to_numeric(df_b['id_operadora'], errors='coerce')
df_b.loc[df_b['id_operadora'] < 0, 'id_operadora'] = np.nan
df_b['id_operadora'] = df_b['id_operadora'].fillna(-1).astype('int64')

print('Beneficiarios apos tratamento:', df_b.shape)
df_b.head()

Beneficiarios apos tratamento: (9800, 6)


,id_beneficiario,sexo,idade,uf,id_operadora,faixa_etaria
0,1,F,36.0,AL,172,31-45
1,2,M,55.0,BA,264,46-60
2,3,M,50.0,CE,106,46-60
3,4,Não Informado,39.0,RS,161,31-45
4,5,F,35.0,RS,18,31-45


### 3.2 Operadoras.csv

In [5]:
print('--- Nulos ---')
print(df_oper_raw.isna().sum())

id_op_num = pd.to_numeric(df_oper_raw['id_operadora'], errors='coerce')
print('\n--- id_operadora ---')
print('Não numéricos:', df_oper_raw.loc[id_op_num.isna(), 'id_operadora'].unique())
print('Negativos:', (id_op_num < 0).sum())
print('Duplicados (como string):', df_oper_raw['id_operadora'].duplicated().sum())

print('\n--- Amostra de nomes (observe o código embutido no final) ---')
print(df_oper_raw['operadora'].head(10).tolist())

--- Nulos ---
id_operadora    11
operadora       11
dtype: int64

--- id_operadora ---
Não numéricos: ['ABC' nan]
Negativos: 5
Duplicados (como string): 35

--- Amostra de nomes (observe o código embutido no final) ---
['Ideal Saúde 112', 'Mais Saúde Benefícios 002', 'Nacional Medical 003', 'Prime Vida Plena 004', 'Integra Assistência 005', 'Viva Planos 006', 'Brasil Serviços Médicos 007', 'Viva Medical 008', 'Regional Benefícios 009', 'Vida Care 010']


**Achados — Operadoras.csv**

O campo `operadora` carrega, ao final do nome, um código numérico de 3 dígitos que corresponde
ao `id_operadora` correto (ex.: `"Ideal Saúde 112"` → id `112`; `"Prime Vida Plena 004"` → id
`4`). Isso vale inclusive para os registros com `id_operadora` corrompido — `"Mais Saúde
Benefícios 002"` tem `id_operadora = -34`, e `"Nacional Medical 003"` tem `id_operadora = ABC`.

**Decisão de tratamento (Operadoras) — reconciliação via nome:**
1. Extrair via regex o código numérico ao final de `operadora` (`\d+$`).
2. Quando `id_operadora` for inválido (nulo, negativo ou não numérico) e o código extraído do
   nome não colidir com nenhum `id_operadora` válido já existente, usar o código do nome como
   `id_operadora` corrigido.
3. Caso não seja possível reconciliar (sem código no nome, ou colisão), o registro é descartado
   da dimensão e qualquer referência a ele (em `Beneficiarios`) cai no sentinela `-1`.
4. Após a reconciliação, remover duplicidade de `id_operadora` mantendo a primeira ocorrência.
5. Acrescentar o registro sentinela `id_operadora = -1, operadora = 'Não Identificada'` para
   suportar as FKs órfãs vindas de `Beneficiarios`/`Atendimentos`.

In [6]:
df_o = df_oper_raw.copy()

codigo_extraido = df_o['operadora'].str.extract(r'(\d{2,})\s*(?:-.*)?$')[0]
id_op_num = pd.to_numeric(df_o['id_operadora'], errors='coerce')
ids_validos = set(id_op_num.dropna().loc[id_op_num.dropna() > 0].astype('int64'))

def reconciliar(id_original, codigo_nome):
    if pd.notna(id_original) and id_original > 0:
        return id_original
    if pd.isna(codigo_nome):
        return np.nan
    candidato = int(codigo_nome)
    if candidato in ids_validos:
        return np.nan  # colisão: não é seguro reatribuir
    return candidato

df_o['id_operadora_tratado'] = [
    reconciliar(id_orig, cod) for id_orig, cod in zip(id_op_num, codigo_extraido)
]

qtd_reconciliados = (
    (df_o['id_operadora_tratado'].notna())
    & ((id_op_num.isna()) | (id_op_num <= 0))
).sum()
qtd_descartados = df_o['id_operadora_tratado'].isna().sum()
print(f'Registros reconciliados via código do nome: {qtd_reconciliados}')
print(f'Registros descartados (sem reconciliação possível): {qtd_descartados}')

df_o = df_o.dropna(subset=['id_operadora_tratado']).copy()
df_o['id_operadora'] = df_o['id_operadora_tratado'].astype('int64')
df_o = df_o.drop(columns=['id_operadora_tratado'])
df_o['operadora'] = df_o['operadora'].str.strip()
df_o = df_o.drop_duplicates(subset='id_operadora', keep='first')

# registro sentinela para FKs órfãs
df_o = pd.concat([
    df_o,
    pd.DataFrame([{'id_operadora': -1, 'operadora': 'Não Identificada'}])
], ignore_index=True)

print('Operadoras apos tratamento:', df_o.shape)
df_o.head()

Registros reconciliados via código do nome: 21
Registros descartados (sem reconciliação possível): 0
Operadoras apos tratamento: (280, 2)


,id_operadora,operadora
0,112,Ideal Saúde 112
1,2,Mais Saúde Benefícios 002
2,3,Nacional Medical 003
3,4,Prime Vida Plena 004
4,5,Integra Assistência 005


### 3.3 Atendimentos.csv

In [7]:
print('--- Nulos ---')
print(df_atend_raw.isna().sum())

print('\n--- Duplicidade ---')
print('Linhas 100% duplicadas:', df_atend_raw.duplicated().sum())
print('id_atendimento duplicado:', df_atend_raw['id_atendimento'].duplicated().sum())

print('\n--- especialidade: exemplos de variação de grafia ---')
print(sorted(df_atend_raw['especialidade'].dropna().str.strip().str.upper().unique())[:10])

id_benef_num = pd.to_numeric(df_atend_raw['id_beneficiario'], errors='coerce')
print('\n--- id_beneficiario (FK) ---')
print('Não numéricos:', df_atend_raw.loc[id_benef_num.isna() & df_atend_raw['id_beneficiario'].notna(), 'id_beneficiario'].unique())

valor_num = pd.to_numeric(df_atend_raw['valor'], errors='coerce')
print('\n--- valor ---')
print('Não numérico (amostra):', df_atend_raw.loc[valor_num.isna() & df_atend_raw['valor'].notna(), 'valor'].unique())
print('Negativos:', (valor_num < 0).sum())

data_dt = pd.to_datetime(df_atend_raw['data_atendimento'], errors='coerce', format='%Y-%m-%d')
print('\n--- data_atendimento ---')
print('Não parseável (formato):', (data_dt.isna() & df_atend_raw['data_atendimento'].notna()).sum())
print('Fora da faixa plausível (<2015 ou >hoje):', ((data_dt < DATA_MIN_VALIDA) | (data_dt > HOJE)).sum())

--- Nulos ---
id_atendimento         0
id_beneficiario     1800
data_atendimento    1800
especialidade       1800
valor               2166
dtype: int64

--- Duplicidade ---
Linhas 100% duplicadas: 967


id_atendimento duplicado: 1800

--- especialidade: exemplos de variação de grafia ---
['ALERGOLOGIA', 'CARDIOLOGIA', 'CIRURGIA GERAL', 'CIRURGIA VASCULAR', 'CLINICA MÉDICA', 'CLÍNICA MÉDICA', 'DERMATOLOGIA', 'ENDOCRINOLOGIA', 'ESPECIALIDADE DESCONHECIDA', 'FISIOTERAPIA']



--- id_beneficiario (FK) ---


Não numéricos: ['ABC']

--- valor ---
Não numérico (amostra): ['erro' 'R$ 250,00' '1.2.3']
Negativos: 1800



--- data_atendimento ---
Não parseável (formato): 1468
Fora da faixa plausível (<2015 ou >hoje): 332


**Achados — Atendimentos.csv**

| Campo | Problema | Qtd. (aprox.) |
|---|---|---|
| (linha inteira) | duplicidade total | 967 |
| id_atendimento | duplicado | 1.800 |
| id_beneficiario / data / especialidade | nulos (mesmo conjunto de linhas) | 1.800 |
| id_beneficiario | não numérico (`ABC`) | 602 |
| id_beneficiario | numérico mas não existe em `Beneficiarios` (órfão) | ~3.110 |
| valor | nulo | 2.166 |
| valor | não numérico (`erro`, `1.2.3`) ou mal formatado (`R$ 250,00`) | — |
| valor | negativo | ~2.168 |
| data_atendimento | formato não parseável ou fora de `[2015-01-01, hoje]` | ~1.630 |
| especialidade | variação de grafia (maiúsculas/minúsculas/acentos/espaços) e placeholder `Especialidade Desconhecida` | — |

As 1.800 linhas com `id_beneficiario`, `data_atendimento` e `especialidade` nulos simultaneamente
não carregam nenhuma informação analítica aproveitável (mesmo tendo `id_atendimento` e às vezes
`valor`) e são descartadas.

**Decisões de tratamento (Atendimentos):**
1. Remover linhas 100% duplicadas.
2. Remover as ~1.800 linhas "vazias" (nulas nos 3 campos-chave acima).
3. `id_atendimento` duplicado remanescente → manter a primeira ocorrência.
4. `valor`: tentar recuperar formatos monetários (`R$ 250,00` → `250.00`); valores não numéricos
   remanescentes (`erro`, `1.2.3`) ou **negativos** (inconsistente para um atendimento) → `NaN` e
   a linha é **excluída da fato**, pois `valor` é a medida central de todas as métricas pedidas
   (valor total, ticket médio) e um atendimento sem valor confiável não deve compor essas somas.
5. `data_atendimento`: fora do intervalo plausível `[2015-01-01, 2026-09-05]` ou não parseável →
   `NaT` e a linha é excluída da fato (sem data não há como popular a dimensão tempo nem a
   granularidade do fato).
6. `especialidade`: normalizar (strip + Title Case); `Especialidade Desconhecida` mantida como
   categoria própria "Não Informada".
7. `id_beneficiario`: não numérico ou órfão → aponta para o sentinela `-1 = Beneficiário Não
   Identificado` (criado na dimensão) em vez de descartar a linha, preservando o valor financeiro
   do atendimento nas métricas agregadas que não dependem do perfil do beneficiário.

Um detalhe adicional apareceu já na normalização de `especialidade`: além de maiúsculas/minúsculas
e espaçamento, há variantes **sem acentuação** que um simples `strip().title()` não unifica
(ex.: `"Clinica Medica"` vs. `"Clínica Médica"`, `"Medicina de Familia"` vs. `"Medicina de
Família"`). O tratamento abaixo normaliza removendo acentos apenas para fins de comparação e
mapeia para uma lista fixa de 25 especialidades com a grafia correta.

In [8]:
df_a = df_atend_raw.drop_duplicates().copy()

# descarta linhas sem nenhuma informação aproveitável
linhas_vazias = df_a['id_beneficiario'].isna() & df_a['data_atendimento'].isna() & df_a['especialidade'].isna()
print('Linhas vazias descartadas:', linhas_vazias.sum())
df_a = df_a.loc[~linhas_vazias].copy()

df_a = df_a.drop_duplicates(subset='id_atendimento', keep='first')
df_a['id_atendimento'] = df_a['id_atendimento'].astype('int64')

# valor: recupera formato monetário BR "R$ 250,00" (ponto = milhar, vírgula = decimal);
# valores que já vêm em formato numérico simples ("241.34") não são tocados.
def limpar_valor(v):
    s = str(v).strip()
    s = re.sub(r'^R\$\s*', '', s)
    if ',' in s:
        s = s.replace('.', '').replace(',', '.')
    return s

valor_limpo = df_a['valor'].apply(limpar_valor)
df_a['valor'] = pd.to_numeric(valor_limpo, errors='coerce')
df_a.loc[df_a['valor'] < 0, 'valor'] = np.nan

# data
df_a['data_atendimento'] = pd.to_datetime(df_a['data_atendimento'], errors='coerce', format='%Y-%m-%d')
fora_da_faixa = (df_a['data_atendimento'] < DATA_MIN_VALIDA) | (df_a['data_atendimento'] > HOJE)
df_a.loc[fora_da_faixa, 'data_atendimento'] = pd.NaT

antes = len(df_a)
df_a = df_a.dropna(subset=['valor', 'data_atendimento']).copy()
print(f'Linhas excluídas por valor/data invalidos: {antes - len(df_a)}')
print(f'Atendimentos com valor = 0 (mantidos; interpretados como isentos, não como erro): {(df_a["valor"] == 0).sum()}')

# especialidade: mapeia para grafia canônica ignorando acentuação/caixa
import unicodedata

def remover_acentos(texto):
    nfkd = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

ESPECIALIDADES_CANONICAS = [
    'Alergologia', 'Cardiologia', 'Cirurgia Geral', 'Cirurgia Vascular', 'Clínica Médica',
    'Dermatologia', 'Endocrinologia', 'Fisioterapia', 'Gastroenterologia', 'Ginecologia',
    'Hematologia', 'Infectologia', 'Medicina de Família', 'Nefrologia', 'Neurologia',
    'Nutrologia', 'Oftalmologia', 'Oncologia', 'Ortopedia', 'Otorrinolaringologia',
    'Pediatria', 'Pneumologia', 'Psiquiatria', 'Reumatologia', 'Urologia',
]
mapa_especialidade_canonica = {remover_acentos(e).upper(): e for e in ESPECIALIDADES_CANONICAS}

chave_especialidade = df_a['especialidade'].fillna('').str.strip().apply(remover_acentos).str.upper()
df_a['especialidade'] = chave_especialidade.map(mapa_especialidade_canonica).fillna('Não Informada')

# id_beneficiario -> FK, com sentinela para invalidos/orfaos
df_a['id_beneficiario'] = pd.to_numeric(df_a['id_beneficiario'], errors='coerce')
orfaos = ~df_a['id_beneficiario'].isin(df_b['id_beneficiario'])
print('id_beneficiario invalido/orfao (aponta p/ sentinela -1):', orfaos.sum())
df_a.loc[orfaos, 'id_beneficiario'] = -1
df_a['id_beneficiario'] = df_a['id_beneficiario'].astype('int64')

print('Atendimentos apos tratamento:', df_a.shape)
df_a.head()

Linhas vazias descartadas: 0


Linhas excluídas por valor/data invalidos: 8272
Atendimentos com valor = 0 (mantidos; interpretados como isentos, não como erro): 368


id_beneficiario invalido/orfao (aponta p/ sentinela -1): 5300
Atendimentos apos tratamento: (89928, 5)


,id_atendimento,id_beneficiario,data_atendimento,especialidade,valor
0,1,452,2024-08-26,Fisioterapia,241.34
1,2,3654,2026-04-05,Clínica Médica,265.27
2,3,1631,2024-07-30,Ortopedia,595.82
3,4,7798,2023-01-17,Gastroenterologia,515.23
4,8325,7145,2023-06-21,Gastroenterologia,325.95


Precisamos também do beneficiário sentinela (`-1`) na dimensão, para as FKs órfãs geradas
acima não quebrarem os relacionamentos do modelo dimensional.

In [9]:
sentinela_benef = pd.DataFrame([{
    'id_beneficiario': -1, 'sexo': 'Não Informado', 'idade': np.nan,
    'faixa_etaria': 'Não Informada', 'uf': 'NI', 'id_operadora': -1
}])
df_b = pd.concat([df_b, sentinela_benef], ignore_index=True)
print('Beneficiarios final:', df_b.shape)

Beneficiarios final: (9801, 6)


## 3.4 Integridade referencial entre arquivos

Além dos problemas internos de cada arquivo, o enunciado pede para avaliar **integridade entre
os arquivos**. Um caso legítimo (não é erro de formatação, é referência inexistente): há
`id_operadora` em `Beneficiarios.csv` que são numéricos e positivos, mas não correspondem a
nenhum `id_operadora` presente em `Operadoras.csv` (mesmo após a reconciliação da seção 3.2).
Esses casos também são redirecionados para o sentinela `-1 = Operadora Não Identificada`.

In [10]:
orfaos_operadora = ~df_b['id_operadora'].isin(df_o['id_operadora'])
print('Beneficiarios com id_operadora inexistente em Operadoras:', orfaos_operadora.sum())
df_b.loc[orfaos_operadora, 'id_operadora'] = -1

Beneficiarios com id_operadora inexistente em Operadoras: 652


## 4. Verificação de integridade referencial pós-tratamento

In [11]:
assert df_a['id_beneficiario'].isin(df_b['id_beneficiario']).all(), 'FK id_beneficiario orfao na fato'
assert df_b['id_operadora'].isin(df_o['id_operadora']).all(), 'FK id_operadora orfao em Beneficiarios'
assert df_a['id_atendimento'].is_unique
assert df_b['id_beneficiario'].is_unique
assert df_o['id_operadora'].is_unique
print('Integridade referencial OK — todas as FKs resolvem para uma PK existente.')

Integridade referencial OK — todas as FKs resolvem para uma PK existente.


## 5. Construção do modelo dimensional (estrela)

- `dim_beneficiario`: id_beneficiario, sexo, idade, faixa_etaria, uf, id_operadora (mantida como
  atributo aqui e desnormalizada na fato — ver `modelagem/modelo_dimensional.md`).
- `dim_operadora`: id_operadora, operadora.
- `dim_especialidade`: id_especialidade (surrogate), especialidade.
- `dim_tempo`: data, ano, trimestre, mes, nome_mes, dia, dia_semana — uma linha por data distinta
  presente nos atendimentos.
- `fato_atendimento`: grão = 1 atendimento. Chaves: id_beneficiario, id_operadora (desnormalizada
  a partir do beneficiário, para permitir "Gastos por operadora" sem *snowflake*), id_especialidade,
  data. Medida: valor.

In [12]:
# dim_especialidade
especialidades = sorted(df_a['especialidade'].unique())
dim_especialidade = pd.DataFrame({
    'id_especialidade': range(1, len(especialidades) + 1),
    'especialidade': especialidades,
})

# dim_tempo
datas = pd.DataFrame({'data': sorted(df_a['data_atendimento'].unique())})
datas['data'] = pd.to_datetime(datas['data'])
dim_tempo = pd.DataFrame({
    'data': datas['data'],
    'ano': datas['data'].dt.year,
    'trimestre': datas['data'].dt.quarter,
    'mes': datas['data'].dt.month,
    'nome_mes': datas['data'].dt.strftime('%B'),
    'dia': datas['data'].dt.day,
    'dia_semana': datas['data'].dt.strftime('%A'),
})

# fato_atendimento: junta especialidade (id) e operadora (via beneficiario)
mapa_especialidade = dict(zip(dim_especialidade['especialidade'], dim_especialidade['id_especialidade']))
mapa_operadora_por_beneficiario = dict(zip(df_b['id_beneficiario'], df_b['id_operadora']))

fato_atendimento = pd.DataFrame({
    'id_atendimento': df_a['id_atendimento'],
    'id_beneficiario': df_a['id_beneficiario'],
    'id_operadora': df_a['id_beneficiario'].map(mapa_operadora_por_beneficiario),
    'id_especialidade': df_a['especialidade'].map(mapa_especialidade),
    'data': df_a['data_atendimento'],
    'valor': df_a['valor'].round(2),
})

dim_beneficiario = df_b[['id_beneficiario', 'sexo', 'idade', 'faixa_etaria', 'uf', 'id_operadora']].copy()
dim_operadora = df_o[['id_operadora', 'operadora']].copy()

print('dim_beneficiario  :', dim_beneficiario.shape)
print('dim_operadora     :', dim_operadora.shape)
print('dim_especialidade :', dim_especialidade.shape)
print('dim_tempo         :', dim_tempo.shape)
print('fato_atendimento  :', fato_atendimento.shape)

dim_beneficiario  : (9801, 6)
dim_operadora     : (280, 2)
dim_especialidade : (26, 2)
dim_tempo         : (1339, 7)
fato_atendimento  : (89928, 6)


## 6. Exportação dos arquivos tratados

In [13]:
dim_beneficiario.to_csv(PASTA_TRATADOS / 'dim_beneficiario.csv', index=False, encoding='utf-8')
dim_operadora.to_csv(PASTA_TRATADOS / 'dim_operadora.csv', index=False, encoding='utf-8')
dim_especialidade.to_csv(PASTA_TRATADOS / 'dim_especialidade.csv', index=False, encoding='utf-8')
dim_tempo.to_csv(PASTA_TRATADOS / 'dim_tempo.csv', index=False, encoding='utf-8')
fato_atendimento.to_csv(PASTA_TRATADOS / 'fato_atendimento.csv', index=False, encoding='utf-8')

print('Arquivos gravados em', PASTA_TRATADOS.resolve())
for f in sorted(PASTA_TRATADOS.glob('*.csv')):
    print(' -', f.name)

Arquivos gravados em E:\Documents\Processo Seletivo\Teste\dados\tratados
 - dim_beneficiario.csv
 - dim_especialidade.csv
 - dim_operadora.csv
 - dim_tempo.csv
 - fato_atendimento.csv


## 7. Resumo do diagnóstico e do tratamento

| Origem | Linhas brutas | Linhas na saída final | Principal motivo da redução |
|---|---:|---:|---|
| Beneficiarios.csv | 10.000 | ver `dim_beneficiario` (inclui sentinela) | duplicidade total + id duplicado |
| Atendimentos.csv | 100.000 | ver `fato_atendimento` | linhas vazias, duplicidade, valor/data inválidos |
| Operadoras.csv | 300 | ver `dim_operadora` (inclui sentinela) | ids não reconciliáveis descartados |

Todas as decisões de tratamento estão documentadas nas seções 3.1–3.3 acima e resumidas no
`README.md` do projeto.

In [14]:
print('Resumo final:')
print('dim_beneficiario  :', len(dim_beneficiario))
print('dim_operadora     :', len(dim_operadora))
print('dim_especialidade :', len(dim_especialidade))
print('dim_tempo         :', len(dim_tempo))
print('fato_atendimento  :', len(fato_atendimento))

Resumo final:
dim_beneficiario  : 9801
dim_operadora     : 280
dim_especialidade : 26
dim_tempo         : 1339
fato_atendimento  : 89928
